In [18]:
import numpy as np
from optikon import full_propositionalization

rng = np.random.default_rng(seed=1)
n = 10000
p = 200
x = rng.multivariate_normal(np.zeros(p), np.eye(p), size=n)
props = full_propositionalization(x)

k = 100
r = 4

conjs=[props[rng.choice(len(props), r, False)] for _ in range(k)]
[len(conj.support_all(x)) for conj in conjs]

/var/folders/zw/qxvhv2ms1rx684818_y1cvl40000gn/T/ipykernel_3428/1154896288.py:7: RuntimeWarning: divide by zero encountered in matmul
  x = rng.multivariate_normal(np.zeros(p), np.eye(p), size=n)
/var/folders/zw/qxvhv2ms1rx684818_y1cvl40000gn/T/ipykernel_3428/1154896288.py:7: RuntimeWarning: overflow encountered in matmul
  x = rng.multivariate_normal(np.zeros(p), np.eye(p), size=n)
/var/folders/zw/qxvhv2ms1rx684818_y1cvl40000gn/T/ipykernel_3428/1154896288.py:7: RuntimeWarning: invalid value encountered in matmul
  x = rng.multivariate_normal(np.zeros(p), np.eye(p), size=n)


[1558,
 39,
 116,
 175,
 63,
 41,
 447,
 445,
 242,
 1184,
 4,
 6,
 2270,
 48,
 61,
 777,
 1941,
 1506,
 12,
 154,
 8,
 70,
 492,
 288,
 112,
 2,
 1972,
 5161,
 0,
 98,
 627,
 5,
 448,
 575,
 2,
 11,
 122,
 3577,
 304,
 568,
 44,
 3172,
 1750,
 3,
 2178,
 1098,
 320,
 0,
 924,
 43,
 917,
 1381,
 4,
 34,
 70,
 1,
 26,
 159,
 294,
 2129,
 150,
 4,
 226,
 10,
 183,
 1271,
 111,
 47,
 1205,
 849,
 3573,
 2284,
 50,
 1312,
 180,
 126,
 317,
 1241,
 3568,
 316,
 28,
 1933,
 6137,
 38,
 115,
 1285,
 631,
 454,
 332,
 1340,
 46,
 1436,
 7,
 4778,
 15,
 51,
 2537,
 1720,
 176,
 94]

In [19]:
%timeit [len(conj.support_all(x)) for conj in conjs]

5.82 ms ± 218 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
from numba import njit

@njit
def support_all(prop, x, q=None):
    """Returns indices of samples satisfying all propositions in q.

    If q is None, all propositions are used (i.e. the entire propositionalisation).

    Args:
        x (ndarray): Input data of shape (n, d).
        q (ndarray or None): Indices of propositions. If None, uses all.

    Returns:
        ndarray: 1D array of indices where all selected propositions hold.
    """
    n = x.shape[0]
    if q is None: q = np.arange(len(prop))
    
    out = np.arange(n)
    k = n
    for j in q:
        l = 0
        for i in range(k):
            idx = out[i]
            if prop.s[j]*x[idx, prop.v[j]] >= prop.t[j]:
                out[l] = idx
                l += 1
        k = l
    
    return out[:k]



In [24]:
[len(support_all(conj, x)) for conj in conjs]

[1558,
 39,
 116,
 175,
 63,
 41,
 447,
 445,
 242,
 1184,
 4,
 6,
 2270,
 48,
 61,
 777,
 1941,
 1506,
 12,
 154,
 8,
 70,
 492,
 288,
 112,
 2,
 1972,
 5161,
 0,
 98,
 627,
 5,
 448,
 575,
 2,
 11,
 122,
 3577,
 304,
 568,
 44,
 3172,
 1750,
 3,
 2178,
 1098,
 320,
 0,
 924,
 43,
 917,
 1381,
 4,
 34,
 70,
 1,
 26,
 159,
 294,
 2129,
 150,
 4,
 226,
 10,
 183,
 1271,
 111,
 47,
 1205,
 849,
 3573,
 2284,
 50,
 1312,
 180,
 126,
 317,
 1241,
 3568,
 316,
 28,
 1933,
 6137,
 38,
 115,
 1285,
 631,
 454,
 332,
 1340,
 46,
 1436,
 7,
 4778,
 15,
 51,
 2537,
 1720,
 176,
 94]

In [25]:
%timeit [len(support_all(conj, x)) for conj in conjs]

8.69 ms ± 34.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [31]:
from numba import prange

@njit
def support_all_parallel(prop, x, q=None):
    n = x.shape[0]
    if q is None: q = np.arange(len(prop))
    
    out = np.ones(n, dtype=np.bool_)
    k = n
    for i in range(k):
        for j in q:
            if prop.s[j]*x[i, prop.v[j]] < prop.t[j]:
                out[i] = False
                break
    
    return np.flatnonzero(out)

[len(support_all_parallel(conj, x)) for conj in conjs]

[1558,
 39,
 116,
 175,
 63,
 41,
 447,
 445,
 242,
 1184,
 4,
 6,
 2270,
 48,
 61,
 777,
 1941,
 1506,
 12,
 154,
 8,
 70,
 492,
 288,
 112,
 2,
 1972,
 5161,
 0,
 98,
 627,
 5,
 448,
 575,
 2,
 11,
 122,
 3577,
 304,
 568,
 44,
 3172,
 1750,
 3,
 2178,
 1098,
 320,
 0,
 924,
 43,
 917,
 1381,
 4,
 34,
 70,
 1,
 26,
 159,
 294,
 2129,
 150,
 4,
 226,
 10,
 183,
 1271,
 111,
 47,
 1205,
 849,
 3573,
 2284,
 50,
 1312,
 180,
 126,
 317,
 1241,
 3568,
 316,
 28,
 1933,
 6137,
 38,
 115,
 1285,
 631,
 454,
 332,
 1340,
 46,
 1436,
 7,
 4778,
 15,
 51,
 2537,
 1720,
 176,
 94]

In [32]:
%timeit [len(support_all_parallel(conj, x)) for conj in conjs]

10.8 ms ± 27.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [35]:
@njit
def support_all_mask(prop, x, q=None):
    n = x.shape[0]
    if q is None: q = np.arange(len(prop))

    mask = np.ones(n, dtype=np.bool_)
    for i in range(q.size):
        j = q[i]
        mask &= prop.s[j] * x[:, prop.v[j]] >= prop.t[j]
    return mask

[np.sum(support_all_mask(conj, x)) for conj in conjs]

[np.int64(1558),
 np.int64(39),
 np.int64(116),
 np.int64(175),
 np.int64(63),
 np.int64(41),
 np.int64(447),
 np.int64(445),
 np.int64(242),
 np.int64(1184),
 np.int64(4),
 np.int64(6),
 np.int64(2270),
 np.int64(48),
 np.int64(61),
 np.int64(777),
 np.int64(1941),
 np.int64(1506),
 np.int64(12),
 np.int64(154),
 np.int64(8),
 np.int64(70),
 np.int64(492),
 np.int64(288),
 np.int64(112),
 np.int64(2),
 np.int64(1972),
 np.int64(5161),
 np.int64(0),
 np.int64(98),
 np.int64(627),
 np.int64(5),
 np.int64(448),
 np.int64(575),
 np.int64(2),
 np.int64(11),
 np.int64(122),
 np.int64(3577),
 np.int64(304),
 np.int64(568),
 np.int64(44),
 np.int64(3172),
 np.int64(1750),
 np.int64(3),
 np.int64(2178),
 np.int64(1098),
 np.int64(320),
 np.int64(0),
 np.int64(924),
 np.int64(43),
 np.int64(917),
 np.int64(1381),
 np.int64(4),
 np.int64(34),
 np.int64(70),
 np.int64(1),
 np.int64(26),
 np.int64(159),
 np.int64(294),
 np.int64(2129),
 np.int64(150),
 np.int64(4),
 np.int64(226),
 np.int64(10),
 

In [36]:
%timeit [np.sum(support_all_mask(conj, x)) for conj in conjs]

5.64 ms ± 310 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
